# Paper 1 — ST-GCN Encoder-Decoder (COMSNETS 2025)
**Sharma et al.** — *Indian Sign Language Recognition and Translation: An Encoder-Decoder Approach*

### Architecture (Paper §IV):
```
Frames → MediaPipe Keypoints (225D: pose+hands) [§IV-A]
  → 4× ST-GCN Block  [§IV-D, Eq.1-2]
  → 3-layer BiLSTM Encoder
  → 4-layer Transformer Decoder
  → Translation output
```
### Evaluation (Paper §VI): BLEU-1/2/3/4 + METEOR
### Training: loss ↓ (down) + **token accuracy % ↑ (up)** shown live

In [ ]:
import os, re, json, math, random
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'text.color':'#e6edf3','axes.titlecolor':'#58a6ff',
    'xtick.color':'#8b949e','grid.color':'#21262d','font.size':11})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE   = r'D:\ISL2\ISL_CSLRT_Corpus'
FRAMES = os.path.join(BASE, 'Frames_Sentence_Level')
KPT    = os.path.join(BASE, 'paper1_kpts')
CKPT   = os.path.join(BASE, 'best_paper1.pth')
PAD, BOS, EOS = '<pad>', '<bos>', '<eos>'
TR, VA, TE = ['1','2','3','4','5'], ['6'], ['7']

torch.manual_seed(42); np.random.seed(42); random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda': print(torch.cuda.get_device_name(0))

## Step 1 — MediaPipe Keypoint Extraction (Paper §IV-A)
> Extracts 33 pose + 21 left hand + 21 right hand = **225D** per frame.
> Paper uses all 1629D (incl. face); we drop face landmarks to reduce noise on small dataset.

In [ ]:
import cv2, mediapipe as mp
from tqdm import tqdm

def tokenize(t):
    if not t: return []
    return [w for w in re.sub(r"[^a-z0-9 ']", ' ', str(t).lower().strip()).split() if w]

f2s = {f: ' '.join(tokenize(f)) for f in sorted(os.listdir(FRAMES))}

exist = sum(len([f for f in os.listdir(os.path.join(KPT,sp)) if f.endswith('.npy')])
    for sp in ['train','val','test'] if os.path.isdir(os.path.join(KPT,sp)))

if exist > 100:
    print(f'✅ Cached: {exist} keypoint files. Skipping extraction.')
else:
    print('Extracting MediaPipe keypoints...')
    for sp in ['train','val','test']: os.makedirs(os.path.join(KPT,sp), exist_ok=True)

    def extract(res):
        d = []
        for part, n in [(res.pose_landmarks,33),(res.left_hand_landmarks,21),(res.right_hand_landmarks,21)]:
            if part: [d.extend([l.x,l.y,l.z]) for l in part.landmark]
            else: d.extend([0.]*n*3)
        return np.array(d, dtype=np.float32)  # 225D

    def normalize(lm):
        lm = lm.reshape(-1, 3)
        try:
            c = (lm[11] + lm[12]) / 2; lm -= c
            s = np.linalg.norm(lm[11] - lm[12])
            if s > 1e-6: lm /= s
        except: pass
        return lm.flatten()

    done = 0
    with mp.solutions.holistic.Holistic(model_complexity=1) as h:
        for sf in tqdm(sorted(os.listdir(FRAMES)), desc='Sentences'):
            for sig in sorted(os.listdir(os.path.join(FRAMES, sf))):
                sp2 = os.path.join(FRAMES, sf, sig)
                if not os.path.isdir(sp2): continue
                split = 'train' if sig in TR else ('val' if sig in VA else 'test')
                safe  = re.sub(r'[^\w]', '_', f'{sf}_s{sig}')[:100]
                no = os.path.join(KPT, split, f'{safe}.npy')
                lo = os.path.join(KPT, split, f'{safe}_label.txt')
                if os.path.exists(no): done += 1; continue
                imgs = [f for f in sorted(os.listdir(sp2)) if f.lower().endswith(('.jpg','.png'))]
                seq = []
                for img in imgs:
                    try:
                        fr = cv2.imread(os.path.join(sp2, img))
                        if fr is None: continue
                        seq.append(normalize(extract(h.process(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))))
                    except: continue
                if not seq: continue
                np.save(no, np.array(seq, dtype=np.float32))
                open(lo, 'w', encoding='utf-8').write(f2s[sf])
                done += 1
    print(f'Done: {done} saved')

for sp in ['train','val','test']:
    d = os.path.join(KPT, sp)
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.endswith('.npy')])
        print(f'  {sp}: {n} files')

## Step 2 — Build Vocabulary

In [ ]:
all_t = []
for sp in ['train','val','test']:
    d = os.path.join(KPT, sp)
    if not os.path.isdir(d): continue
    for f in os.listdir(d):
        if f.endswith('_label.txt'):
            all_t.extend(tokenize(open(os.path.join(d,f), encoding='utf-8').read()))

vocab = {PAD:0, BOS:1, EOS:2, '<blank>':3}
for i, t in enumerate(sorted(set(all_t)), 4): vocab[t] = i
i2t = {v:k for k,v in vocab.items()}
V = len(vocab)
PI = vocab[PAD]; BI = vocab[BOS]; EI = vocab[EOS]; BK = vocab['<blank>']

json.dump(vocab, open(os.path.join(BASE,'gloss_vocab.json'),'w'), indent=2)
print(f'Vocabulary: {V} tokens')
print(f'Sample: {list(vocab.keys())[4:14]}')

## Step 3 — Dataset with Augmentation

In [ ]:
class D1(Dataset):
    def __init__(self, dr, mn=300, tr=False):
        self.s, self.mn, self.tr, self.fd = [], mn, tr, None
        for f in sorted(os.listdir(dr)):
            if not f.endswith('.npy'): continue
            lb = os.path.join(dr, f[:-4]+'_label.txt')
            if not os.path.exists(lb): continue
            txt = open(lb, encoding='utf-8').read().strip()
            tks = tokenize(txt)
            if not tks: continue
            gi = [vocab.get(t, PI) for t in tks]
            ti = [BI] + gi + [EI]
            self.s.append((os.path.join(dr, f), gi, ti, txt))
        if self.s: self.fd = np.load(self.s[0][0]).shape[1]
        print(f'  {dr.split(os.sep)[-1]:6s}: {len(self.s)} samples  dim={self.fd}')

    def __len__(self): return len(self.s)

    def __getitem__(self, i):
        p, gi, ti, _ = self.s[i]
        x = np.nan_to_num(np.load(p).astype(np.float32))
        if self.tr:
            if random.random() < 0.5: x += np.random.randn(*x.shape).astype(np.float32) * 0.008
            if random.random() < 0.4:
                T = x.shape[0]; r = random.uniform(0.85, 1.15)
                idx = np.round(np.linspace(0, T-1, max(int(T*r), 4))).astype(int).clip(0, T-1)
                x = x[idx]
        if x.shape[0] > self.mn:
            idx = np.round(np.linspace(0, x.shape[0]-1, self.mn)).astype(int)
            x = x[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(gi, dtype=torch.long), torch.tensor(ti, dtype=torch.long)


def collate(batch):
    xs, gs, ts = zip(*batch)
    xl = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
    gl = torch.tensor([g.shape[0] for g in gs], dtype=torch.long)
    return (pad_sequence(xs, batch_first=True),
            pad_sequence(gs, batch_first=True, padding_value=PI),
            pad_sequence(ts, batch_first=True, padding_value=PI),
            xl, gl)


print('Loading datasets...')
tr_ds = D1(os.path.join(KPT, 'train'), tr=True)
vl_ds = D1(os.path.join(KPT, 'val'),   tr=False)
ID = tr_ds.fd

assert ID is not None, 'No training samples found! Run Step 1 first.'
assert len(vl_ds) > 0, 'No val samples found!'

tr_ld = DataLoader(tr_ds, batch_size=8, shuffle=True,  collate_fn=collate, num_workers=0)
vl_ld = DataLoader(vl_ds, batch_size=8, shuffle=False, collate_fn=collate, num_workers=0)
print(f'Input dim: {ID} | Vocab: {V}')
print(f'Train: {len(tr_ds)} | Val: {len(vl_ds)}')

## Step 4 — ST-GCN Encoder-Decoder Model (Paper §IV-D)

**Graph update rule — Eq(1) from paper:**
> `H^(l+1) = σ( D̃^(-½) Ã D̃^(-½) H^(l) W^(l) )`

**Node aggregation — Eq(2) from paper:**
> `h^(k)_i = σ( Σ_{j∈N(i)} f(h^(k-1)_i, h^(k-1)_j, e_ij) )`

Implemented as: spatial FC (W^l) + temporal Conv1D + residual connection.

In [ ]:
class STGCN(nn.Module):
    """Spatial-Temporal GCN — Paper 1 §IV-D, Eq(1)(2)"""
    def __init__(self, ic, oc, tk=3, p=0.3):
        super().__init__()
        self.sp = nn.Linear(ic, oc)                           # W^(l): spatial weights
        self.tc = nn.Conv1d(oc, oc, tk, padding=tk//2)        # temporal convolution
        self.bn = nn.BatchNorm1d(oc)
        self.act = nn.GELU()
        self.dp  = nn.Dropout(p)
        self.rs  = nn.Linear(ic, oc, bias=False) if ic != oc else nn.Identity()

    def forward(self, x):                                      # [B, T, C_in]
        h = self.act(self.sp(x))                               # Spatial GCN: σ(H W^(l))
        h = self.bn(self.tc(h.permute(0,2,1))).permute(0,2,1) # Temporal conv + BN
        return self.dp(h + self.rs(x))                         # Residual connection


class Paper1Model(nn.Module):
    """
    Full Encoder-Decoder model (Paper 1):
    4×STGCN → 3×BiLSTM Encoder → CTC head (recognition)
             → 4×Transformer Decoder (translation/captioning)
    """
    def __init__(self, idim, vs, d=256, p=0.3):
        super().__init__()
        # Encoder: ST-GCN + BiLSTM (Paper §IV)
        self.enc = nn.Sequential(
            STGCN(idim, 128, p=p),
            STGCN(128,  256, p=p),
            STGCN(256,  256, p=p),
            STGCN(256,  d,   p=p),
        )
        self.lstm = nn.LSTM(d, d//2, 3, batch_first=True, bidirectional=True, dropout=p)
        self.ln   = nn.LayerNorm(d)
        self.ctc  = nn.Linear(d, vs)  # CTC for recognition (auxiliary)

        # Decoder: Transformer (Paper §IV)
        self.emb = nn.Embedding(vs, d, padding_idx=PI)
        self.pe  = nn.Embedding(80, d)
        dl = nn.TransformerDecoderLayer(d, 8, 1024, p, batch_first=True, norm_first=True)
        self.dec = nn.TransformerDecoder(dl, 4)
        self.out = nn.Linear(d, vs)
        self.dp  = nn.Dropout(p)

    def encode(self, x):
        x = self.enc(x)         # ST-GCN spatial-temporal features
        x, _ = self.lstm(x)     # BiLSTM temporal context
        return self.ln(x)       # [B, T, d]

    def decode(self, t, m):
        pos = torch.arange(t.shape[1], device=t.device).unsqueeze(0)
        x   = self.dp(self.emb(t) + self.pe(pos))
        msk = nn.Transformer.generate_square_subsequent_mask(t.shape[1], device=t.device)
        pad = (t == PI)
        return self.out(self.dec(x, m, tgt_mask=msk, tgt_key_padding_mask=pad))

    def forward(self, x, t):
        m = self.encode(x)
        return self.ctc(m), self.decode(t[:, :-1], m)


model = Paper1Model(ID, V).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'ST-GCN Encoder-Decoder  |  Params: {params:,}')
print(f'Encoder: 4×STGCN + 3×BiLSTM(256)  |  Decoder: 4×TransformerDecoder')

## Step 5 — Training
Shows **both Loss (↓) and Token Accuracy % (↑)** every epoch.

In [ ]:
EP = 100; LR = 3e-4; PAT = 20

ctc_c = nn.CTCLoss(blank=BK, zero_infinity=True, reduction='mean')
ce_c  = nn.CrossEntropyLoss(ignore_index=PI, label_smoothing=0.1)
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sch   = torch.optim.lr_scheduler.LambdaLR(opt,
    lambda e: (e+1)/10 if e < 10 else 0.5*(1+math.cos(math.pi*(e-10)/max(EP-10,1))))


def token_acc(logits, targets):
    """Token-level accuracy ignoring PAD positions."""
    preds = logits.argmax(-1)
    mask  = (targets != PI)
    return ((preds == targets) & mask).sum().item(), mask.sum().item()


def run(ld, tr=True):
    model.train(tr)
    tot, n, correct, total = 0.0, 0, 0, 0
    for xs, gs, ts, xl, gl in ld:
        xs, gs, ts = xs.to(DEVICE), gs.to(DEVICE), ts.to(DEVICE)
        if tr: opt.zero_grad()
        with torch.set_grad_enabled(tr):
            cl, dl = model(xs, ts)
            # CTC loss — recognition (Paper 1, auxiliary)
            lp  = F.log_softmax(cl, dim=-1).permute(1,0,2)
            lc  = ctc_c(lp, gs, xl.clamp(max=lp.shape[0]).to(DEVICE), gl.to(DEVICE))
            # CE loss — translation/captioning
            tgt_out = ts[:, 1:]
            le  = ce_c(dl.reshape(-1, V), tgt_out.reshape(-1).to(DEVICE))
            loss = le + 0.2 * lc
        if not (torch.isnan(loss) or torch.isinf(loss)):
            if tr:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            tot += loss.item(); n += 1
            c, tot_c = token_acc(dl, tgt_out)
            correct += c; total += tot_c
    acc = correct / max(total, 1) * 100
    return (tot/n, acc) if n > 0 else (float('inf'), 0.0)


th, vh, ta, va = [], [], [], []
bv, ni = float('inf'), 0

print(f'Training {EP} epochs | CE + 0.2×CTC | label_smooth=0.1')
print(f'{"Ep":>4}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Loss":>9}  {"Val Acc":>8}')
print('─' * 57)

for ep in range(1, EP+1):
    tl, tacc = run(tr_ld, True)
    vl, vacc = run(vl_ld, False)
    sch.step()
    th.append(tl); vh.append(vl); ta.append(tacc); va.append(vacc)
    s = ''
    if vl < bv:
        bv=vl; ni=0
        torch.save({'ep':ep,'m':model.state_dict(),'v':vocab,'l':vl}, CKPT)
        s = '  ✅'
    else: ni += 1
    print(f'Ep {ep:3d}/{EP}  loss={tl:.4f}  acc={tacc:5.1f}%  vl={vl:.4f}  vacc={vacc:5.1f}%{s}')
    if ni >= PAT: print('Early stop'); break

print(f'\nBest val loss: {bv:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Paper 1: ST-GCN Encoder-Decoder Training', color='#58a6ff', fontsize=14)

axes[0].plot(th, color='#3fb950', lw=2, label='Train Loss')
axes[0].plot(vh, color='#f85149', lw=2, label='Val Loss')
axes[0].set_title('Loss ↓ (lower = better)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ta, color='#3fb950', lw=2, label='Train Acc %')
axes[1].plot(va, color='#f85149', lw=2, label='Val Acc %')
axes[1].set_title('Token Accuracy % ↑ (higher = better)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE, 'paper1_training.png'), dpi=150)
plt.show()

## Step 6 — Evaluation: BLEU-1/2/3/4 + METEOR (Paper §VI)

In [ ]:
ckpt = torch.load(CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['m'])
print(f'Loaded epoch={ckpt["ep"]}  val_loss={ckpt["l"]:.4f}')


def greedy_gen(xf, mx=25):
    model.eval()
    with torch.no_grad():
        mem = model.encode(torch.tensor(xf).unsqueeze(0).to(DEVICE))
        gen = [BI]
        for _ in range(mx):
            t   = torch.tensor([gen], dtype=torch.long, device=DEVICE)
            nxt = model.decode(t, mem)[0, -1].argmax().item()
            if nxt == EI: break
            gen.append(nxt)
    return [i2t.get(i, '?') for i in gen[1:]]


def bleu_n(ref, hyp, n):
    if len(hyp) < n: return 0.0
    rn = set(zip(*[ref[i:] for i in range(n)]))
    hn = [tuple(hyp[i:i+n]) for i in range(len(hyp)-n+1)]
    return sum(1 for g in hn if g in rn) / max(len(hn), 1)

def meteor(ref, hyp):
    m = len(set(ref) & set(hyp))
    if not m: return 0.0
    p = m / max(len(hyp), 1)
    r = m / max(len(ref), 1)
    return 10 * p * r / max(9*p + r, 1e-8)

def seq_acc(ref, hyp):
    return 1.0 if ref == hyp else 0.0


b1, b2, b3, b4, mt, sa, ex = [], [], [], [], [], [], []

for np_, gi, ti, lb in vl_ds.s:
    x    = np.nan_to_num(np.load(np_).astype(np.float32))
    pred = greedy_gen(x)
    ref  = tokenize(lb)
    b1.append(bleu_n(ref, pred, 1)); b2.append(bleu_n(ref, pred, 2))
    b3.append(bleu_n(ref, pred, 3)); b4.append(bleu_n(ref, pred, 4))
    mt.append(meteor(ref, pred));    sa.append(seq_acc(ref, pred))
    if len(ex) < 8: ex.append((lb, ' '.join(pred) or '<empty>'))

print(f'\n{"═"*55}')
print(f'  BLEU-1       : {np.mean(b1)*100:.2f}%')
print(f'  BLEU-2       : {np.mean(b2)*100:.2f}%')
print(f'  BLEU-3       : {np.mean(b3)*100:.2f}%')
print(f'  BLEU-4       : {np.mean(b4)*100:.2f}%')
print(f'  METEOR       : {np.mean(mt)*100:.2f}%')
print(f'  Sequence Acc : {np.mean(sa)*100:.2f}%  (exact match)')
print(f'{"═"*55}\n')
for r, h in ex:
    print(f'REF: {r}')
    print(f'HYP: {h}\n')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Paper 1: Evaluation Metrics (Paper §VI)', color='#58a6ff', fontsize=14)

# BLEU chart
bvals = [np.mean(b1)*100, np.mean(b2)*100, np.mean(b3)*100, np.mean(b4)*100]
bars = axes[0].bar(['BLEU-1','BLEU-2','BLEU-3','BLEU-4'], bvals,
    color=['#58a6ff','#3fb950','#ffa657','#f85149'], alpha=0.85, width=0.5)
for bar, v in zip(bars, bvals):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.1f}%',
                 ha='center', color='white', fontsize=11, fontweight='bold')
axes[0].set_title('BLEU Scores (Paper §VI metric)'); axes[0].grid(True, alpha=0.3)

# METEOR + Seq Acc
mvals = [np.mean(mt)*100, np.mean(sa)*100]
bars2 = axes[1].bar(['METEOR','Sequence Acc'], mvals, color=['#d2a8ff','#79c0ff'], alpha=0.85, width=0.4)
for bar, v in zip(bars2, mvals):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.1f}%',
                 ha='center', color='white', fontsize=13, fontweight='bold')
axes[1].set_title('METEOR + Exact Sequence Accuracy'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE, 'paper1_eval.png'), dpi=150)
plt.show()